# Audio spoof detection — implementation plan

PPT proposes Mel-spectrogram + ResNet compared with fine-tuned Wav2Vec2. Start with ASVspoof 2019 LA and WaveFake for training/validation; reserve the In-the-Wild audio set and Deepfake-Eval-2024 for evaluation as permitted.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
BASE=Path('/content/drive/MyDrive/VeriSight_AI_Storage')
print('Audio data root:', BASE/'datasets/audio')

## Preprocessing contract
- Convert audio to 16 kHz mono using ffmpeg.
- Keep original sample IDs and labels in a manifest.
- Split by speaker/source recording where metadata allows; avoid speaker leakage.
- Create log-Mel spectrograms for the ResNet baseline.
- Compare with Wav2Vec2 only after the baseline pipeline and evaluation are reproducible.
- Metrics: accuracy, precision, recall, F1, ROC-AUC and EER.

This notebook does not invent an audio score or pretend the audio model is trained.

In [ ]:
import pandas as pd
manifest=BASE/'manifests/audio_samples.csv'
if manifest.exists():
    df=pd.read_csv(manifest)
    print(df.head())
    assert {'path','label','group_id','split','dataset'}.issubset(df.columns)
    assert (df.groupby('group_id').split.nunique()<=1).all(), 'Group leakage detected'
else:
    print('No audio manifest yet. Complete licensed data preparation first.')